# 第 19 课：NumPy 副本和视图

NumPy 中的赋值、切片、`view()` 和 `copy()` 对内存的处理方式不同。掌握它们，才能判断修改一个数组是否会影响另一个数组。核心区别是：视图共享底层数据，副本拥有独立数据。

In [1]:
import numpy as np

print('NumPy 版本：', np.__version__)

NumPy 版本： 2.5.3


## 1. 简单赋值：同一个数组对象

`b = a` 不会创建新数组，`a` 和 `b` 指向同一个对象。修改任意一个变量，另一个变量也会看到变化；`a is b` 的结果为 `True`。

In [2]:
a = np.array([1, 2, 3, 4])
b = a

print('a is b：', a is b)
b[0] = 100
print('修改 b 后的 a：', a)
print('a 与 b 的 data 地址是否相同：', a.ctypes.data == b.ctypes.data)

a is b： True
修改 b 后的 a： [100   2   3   4]
a 与 b 的 data 地址是否相同： True


## 2. 切片和 `view()`：共享数据的视图

NumPy 数组切片通常返回视图，而不是副本。`view()` 也会创建一个新的数组对象，但它仍与原数组共享底层数据。因此修改视图中的元素会修改原数组。可以用 `.base` 检查它是否引用了另一个数组。

In [3]:
original = np.arange(6)
slice_view = original[2:5]
explicit_view = original.view()

slice_view[0] = 200
explicit_view[1] = 300
print('原数组：', original)
print('切片视图：', slice_view)
print('显式视图：', explicit_view)
print('切片是否有 base：', slice_view.base is not None)
print('视图与原数组是否共享地址：', explicit_view.ctypes.data == original.ctypes.data)

原数组： [  0 300 200   3   4   5]
切片视图： [200   3   4]
显式视图： [  0 300 200   3   4   5]
切片是否有 base： True
视图与原数组是否共享地址： True


## 3. `copy()`：独立副本

`copy()` 会复制数据并分配新的内存。修改副本不会改变原数组，适合需要独立保存、独立处理的数据。`copy()` 对普通数值数组是独立副本；包含对象的复杂嵌套结构还需额外考虑对象本身是否共享。

In [8]:
source = np.array([[10, 20], [30, 40]])
clone = source.copy()

clone[0, 0] = 999
print('原数组：\n', source)
print('副本：\n', clone)
print('source is clone：', source is clone)
print('是否共享内存：', np.shares_memory(source, clone))
print('副本的 base：', clone.base)

原数组：
 [[10 20]
 [30 40]]
副本：
 [[999  20]
 [ 30  40]]
source is clone： False
是否共享内存： False
副本的 base： None


## 4. 形状视图与副本的对比

`reshape` 在可能的情况下返回视图，改变结果中的元素可能影响原数组；`flatten` 总是返回副本；`ravel` 尽量返回视图，但在无法共享内存时可能返回副本。需要明确独立数据时，优先使用 `flatten()` 或 `.copy()`。

In [5]:
data = np.arange(6)
reshaped = data.reshape(2, 3)
flattened = reshaped.flatten()
raveled = reshaped.ravel()

reshaped[0, 0] = 50
flattened[0] = 60
raveled[1] = 70
print('data：', data)
print('flatten 结果：', flattened)
print('ravel 结果：', raveled)
print('data 与 flattened 共享内存：', np.shares_memory(data, flattened))
print('data 与 raveled 共享内存：', np.shares_memory(data, raveled))

data： [50 70  2  3  4  5]
flatten 结果： [60  1  2  3  4  5]
ravel 结果： [50 70  2  3  4  5]
data 与 flattened 共享内存： False
data 与 raveled 共享内存： True


## 5. 用 `np.shares_memory` 判断

不要只根据 `id()` 判断数据是否共享：视图本身是不同的数组对象，但可能共享底层内存。`np.shares_memory(x, y)` 检查两个数组是否共享内存；`np.may_share_memory` 是更快但可能保守的判断。

In [6]:
x = np.arange(10)
y = x[::2]
z = x.copy()

print('x 与 y 共享内存：', np.shares_memory(x, y))
print('x 与 z 共享内存：', np.shares_memory(x, z))
y[:] = -1
print('修改 y 后 x：', x)

x 与 y 共享内存： True
x 与 z 共享内存： False
修改 y 后 x： [-1  1 -1  3 -1  5 -1  7 -1  9]


## 本节练习

请完成以下任务：

1. 创建 `a = np.arange(8)`；2. 创建 `view = a[::2]`，把视图中的元素全部改为 `-1`，观察 `a`；3. 创建 `copy = a.copy()`，修改 `copy[0]`，确认 `a` 不变；4. 用 `np.shares_memory` 分别判断 `a` 与 `view`、`a` 与 `copy` 是否共享内存。

In [13]:
a = np.arange(8)
view = a[::2]
view.fill(-1)
print(view)
print(a)
copy = a.copy()
copy[0] = 3
print(copy)
print(a)
print(np.shares_memory(a,view))
print(np.shares_memory(a,copy))

[-1 -1 -1 -1]
[-1  1 -1  3 -1  5 -1  7]
[ 3  1 -1  3 -1  5 -1  7]
[-1  1 -1  3 -1  5 -1  7]
True
False


In [12]:
a = np.arange(8)
view = a[::2]
view[:] = -1
copy = a.copy()
copy[0] = 999

print('修改视图后的 a：', a)
print('修改后的 copy：', copy)
print('a 与 view 共享内存：', np.shares_memory(a, view))
print('a 与 copy 共享内存：', np.shares_memory(a, copy))

修改视图后的 a： [-1  1 -1  3 -1  5 -1  7]
修改后的 copy： [999   1  -1   3  -1   5  -1   7]
a 与 view 共享内存： True
a 与 copy 共享内存： False


### 小结

简单赋值共享同一个数组对象；切片和 `view()` 通常共享底层内存；`copy()` 创建独立副本。修改数据前先判断是否共享内存，可以避免难以发现的连锁修改。下一课将进入 NumPy 矩阵库。